# Discord account token — phone-only recovery (Kaggle)

No laptop needed. This notebook runs a headless browser, you approve the
login **from the Discord app on this same phone**, and the token is printed
for you to copy.

**Before you start**
1. Make sure this notebook is **Private** (Kaggle default — check the shield icon).
2. Keep this tab open and have the Discord app on hand.
3. When you're done, **delete this notebook** — the token stays in the cell
   outputs until the notebook is deleted.

**Cells**: 1 = install (5-8 min, run once) · 2 = open Discord · 3 = get the
QR link · 4 = catch the token · 5 = fallback (password, only if needed) ·
6 = cleanup.

Honest caveat: Discord sometimes puts a captcha on logins from datacenter
machines like Kaggle. If a screenshot below shows a captcha or a risk
check, this path is blocked — the only alternative is a real computer
(a public library one works; it takes 30 seconds).

In [ ]:
# CELL 1 — install the headless browser (takes ~5-8 minutes, run once)
!pip install -q playwright opencv-python-headless
!python -m playwright install --with-deps chromium 2>&1 | tail -2
print('READY — run the next cell')

In [ ]:
# CELL 2 — open Discord in the headless browser and show you the page
import time
from IPython.display import Image
from playwright.sync_api import sync_playwright

pw = sync_playwright().start()
browser = pw.chromium.launch(
    headless=True,
    args=["--no-sandbox", "--disable-blink-features=AutomationControlled"],
)
ctx = browser.new_context(
    viewport={"width": 1280, "height": 900},
    user_agent=(
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    ),
    locale="en-US",
)
page = ctx.new_page()
page.goto("https://discord.com/app", wait_until="domcontentloaded", timeout=60000)
time.sleep(6)
display(Image(data=page.screenshot()))
print("URL now:", page.url)

In [ ]:
# CELL 3 — decode the login QR and give you a link to open on THIS phone
import cv2
import numpy as np

img = cv2.imdecode(np.frombuffer(page.screenshot(), np.uint8), cv2.IMREAD_COLOR)
det = cv2.QRCodeDetector()
qr_url, _pts = det.detectAndDecode(img)[0], None
if not qr_url:  # the QR sits on the right half of the login screen
    h, w = img.shape[:2]
    for sub in (img[:, w // 2:], img[: h // 2, :], img[:, : w // 2:]):
        data, _, _ = det.detectAndDecode(sub)
        if data:
            qr_url = data
            break

if qr_url:
    if qr_url.startswith("discord://"):
        print("ANDROID — copy the WHOLE line below into Chrome's address bar and press go:")
        print()
        print("intent:" + qr_url + ";package=com.discord#Intent;end")
    elif qr_url.startswith("http"):
        print("Copy this URL into your phone browser:")
        print()
        print(qr_url)
    else:
        print("QR payload:", qr_url[:200])
    print()
    print("Your Discord app should now pop up a 'log in to this device' prompt —")
    print("APPROVE IT (enter your 2FA code there if you have it). Then run cell 4.")
else:
    print("No QR code found on the page. Look at the screenshot from cell 2:")
    print("  - email/password form visible  -> use the CELL 5 password fallback")
    print("  - captcha / risk check         -> this path is blocked; use a real computer")
    print("  - already logged in?!          -> skip to cell 4")

In [ ]:
# CELL 4 — wait (up to 10 min) for the login to land, then print the token
token = ""
deadline = time.time() + 600
while time.time() < deadline:
    token = ctx.evaluate("() => window.localStorage.getItem('token')") or ""
    if token:
        break
    time.sleep(3)

if token:
    print("DISCORD TOKEN (copy the whole string):")
    print()
    print(token)
    print()
    print("--- put these two lines in ~/.nomorals/.env where you run nm ---")
    print("NM_CHAT_DISCORD_ENABLED = true")
    print("NM_CHAT_DISCORD_TOKEN   = " + token)
    print("--------------------------------------------------------------------")
    print()
    print("NOW: run cell 6, then DELETE this notebook (the token lives in these outputs).")
else:
    print("No token yet. Did you approve the login prompt in the Discord app?")
    print("If yes, run this cell again. If the page shows a captcha, stop — see the caveat.")

In [ ]:
# CELL 5 — FALLBACK: password login via a Kaggle secret (only if the QR path failed)
# First, on this phone: kaggle.com -> your avatar -> Settings -> Secrets -> New secret
#     name:  DISCORD_LOGIN
#     value: yourdiscordemail|yourpassword     (one vertical bar, no spaces)
# Then: Kernel -> Restart & Clear Output  (so Kaggle injects the secret),
#       run cells 1-2 again, then THIS cell.
from kaggle import secrets

creds = secrets.secrets().get("DISCORD_LOGIN", "")
if not creds or "|" not in creds:
    print("No DISCORD_LOGIN secret yet — create it on kaggle.com (Settings -> Secrets),")
    print("restart the session, run cells 1-2, then this cell.")
else:
    email, password = creds.split("|", 1)
    login = ctx.new_page()
    login.goto("https://discord.com/login", wait_until="domcontentloaded", timeout=60000)
    time.sleep(3)
    try:
        login.fill("input[name='email']", email)
        login.press("input[name='email']", "Enter")
        time.sleep(5)
        login.fill("input[name='password']", password)
        login.press("input[name='password']", "Enter")
        time.sleep(10)
    except Exception as exc:  # noqa: BLE001
        print("form fill failed:", exc)
    display(Image(data=login.screenshot()))
    tok = ctx.evaluate("() => window.localStorage.getItem('token')") or ""
    if tok:
        print("TOKEN:", tok)
        print("put it in .env as in cell 4, then DELETE this notebook and the DISCORD_LOGIN secret.")
    else:
        print("Look at the screenshot — 2FA code screen? Then the QR path (cells 2-4) is")
        print("your better bet, because the 2FA code gets typed in the app, not here.")
        print("Captcha/risk check? This path is blocked; use a real computer.")

In [ ]:
# CELL 6 — cleanup
try:
    browser.close()
    pw.stop()
    print("cleaned up — now DELETE this notebook")
except Exception:  # noqa: BLE001
    print("already closed")